In [31]:
import numpy as np
import pandas as pd
import pyfastx
from utils import parse_vadr_cds_tbl, format_cds_coordinates
import matplotlib.pyplot as plt
from collections import defaultdict
import matplotlib.patches as mpatches
import re
from viz_sequence_v2 import plot_dashboard_grid
import os
from scipy.spatial.distance import cdist, cosine
from sklearn.preprocessing import normalize

In [32]:
df_01_PB2 = pd.read_csv('H5Nx_NorthAmerica_Metadata/01_PB2/H5Nx_1_PB2_ID_Filtered.csv')

In [33]:
df_01_PB2.shape

(10382, 42)

In [34]:
df_01_PB2.index

RangeIndex(start=0, stop=10382, step=1)

In [35]:
stop_codons = {'TAA', 'TAG', 'TGA'}

In [36]:
fa_01_PB2 = pyfastx.Fasta('H5Nx_NorthAmerica_Sequences/01_PB2_H5Nx_North_America.fasta', build_index=True)

In [37]:
cds_tbl_path = f'H5Nx_NorthAmerica_VADR/01_PB2_VADR/01_PB2_VADR.vadr.pass.tbl'
parsed_cds_tbl_data = parse_vadr_cds_tbl(cds_tbl_path)
cds_coords_dict = format_cds_coordinates(parsed_cds_tbl_data)

In [38]:
def batch_cosine_dist(X_query, ref_vector, batch_size=2000):
    """
    Computes cosine distance between many query vectors and a single reference vector.
    Optimization: Uses dot product of normalized vectors instead of cdist.
    
    Args:
        X_query (np.ndarray): Shape (N, features)
        ref_vector (np.ndarray): Shape (features,) or (1, features)
        batch_size (int): Size of chunks (less critical here but good for consistency)
        
    Returns:
        np.ndarray: Array of cosine distances of shape (N,)
    """
    # 1. Ensure ref_vector is shape (1, features)
    ref_vector = ref_vector.reshape(1, -1)
    
    ref_norm = normalize(ref_vector, norm='l2', axis=1)
    
    X_query_norm = normalize(X_query, norm='l2', axis=1)
    
    similarities = X_query_norm.dot(ref_norm.T).flatten()

    similarities = np.clip(similarities, -1.0, 1.0)
    distances = 1.0 - similarities
    
    return distances

In [39]:
def get_codon(sequence, cds_coordinates, codon_pos, codon_start=1):
    """Count codons in the given coding sequence"""
    full_cds_sequence = ""

    # Step 1: Correctly parse ALL start and end coordinates from the string.
    coords = re.findall(r'[<>]?(\d+)\.\.[<>]?(\d+)', cds_coordinates)
    if not coords:
        raise ValueError(f"Could not parse CDS coordinates from string: {cds_coordinates}")

    # Step 2: Build the complete CDS by concatenating all exon parts.
    for start, end in coords:
        start_index, end_index = int(start) - 1, int(end)
        full_cds_sequence += sequence[start_index:end_index]

    # Step 3: Apply the codon_start offset to the ASSEMBLED sequence.
    offset = int(codon_start) - 1
    trimmed_sequence = full_cds_sequence[offset:]

    # Step 4: Process the final sequence
    remainder = len(trimmed_sequence) % 3
    seq_length = len(trimmed_sequence) - remainder

    for pos, codon_idx in enumerate(range(0, seq_length, 3)):
        codon = trimmed_sequence[codon_idx:codon_idx+3]
        if pos+1 == codon_pos:
            return codon
    return ''


In [40]:
df_01_PB2['Codon_Pos_627'] = ''
df_01_PB2['Codon_Pos_701'] = ''

In [41]:
codon_pos_627_dict = {}
codon_pos_701_dict = {}
for index, row in df_01_PB2.iterrows():
    seq_id = row['1_PB2_ID']
    sequence=fa_01_PB2[seq_id].seq.upper()
    cds_coords = cds_coords_dict[seq_id]['polymerase PB2']
    extracted_coords = cds_coords['coordinate']
    codon_start = cds_coords.get('codon_start', 1)
    codon_627 = get_codon(sequence, extracted_coords, 627, codon_start)
    codon_pos_627_dict[seq_id] = codon_627
    
    codon_701 = get_codon(sequence, extracted_coords, 701, codon_start)
    codon_pos_701_dict[seq_id] = codon_701

In [42]:
df_01_PB2['Codon_Pos_627'] = df_01_PB2['1_PB2_ID'].apply(lambda x: codon_pos_627_dict[x])
df_01_PB2['Codon_Pos_701'] = df_01_PB2['1_PB2_ID'].apply(lambda x: codon_pos_701_dict[x])

In [43]:
df_01_PB2['Host_ID'].value_counts(dropna=False)

Host_ID
1    8451
2    1903
0      28
Name: count, dtype: int64

In [44]:
df_01_PB2['Codon_Pos_627'].value_counts(dropna=False).head(10)

Codon_Pos_627
GAA    9697
AAA     184
AGA     151
GAG      56
GTG      51
CTG      36
ATG      27
CAA      25
TCT      23
GGC      23
Name: count, dtype: int64

In [45]:
df_01_PB2['Codon_Pos_701'].value_counts(dropna=False).head(10)

Codon_Pos_701
GAC    9860
TAT     132
AGC      65
AAC      63
GGA      29
GAG      27
GAT      25
CTA      22
ATC      21
AAG      20
Name: count, dtype: int64

In [46]:
matching_index = df_01_PB2[df_01_PB2['Codon_Pos_701'].isin(['AAC', 'AAT'])].index

In [47]:
len(matching_index)

81

In [48]:
df_01_PB2_701 = df_01_PB2.loc[matching_index]

In [49]:
df_01_PB2_701.index

Index([   21,   137,   304,   307,   308,   434,   533,   542,   543,   894,
         935,   991,  1010,  1138,  1335,  1362,  1383,  1399,  1446,  1469,
        1508,  1573,  1776,  2015,  2338,  2444,  2552,  4224,  4739,  4779,
        4815,  4873,  4919,  5149,  5347,  5394,  5462,  5741,  6017,  6100,
        6134,  6156,  6192,  6690,  6755,  6872,  7073,  7074,  7128,  7182,
        7183,  7683,  7763,  7780,  7809,  7827,  7883,  8002,  8084,  8094,
        8108,  8207,  8233,  8275,  8329,  8381,  8393,  8670,  8860,  9099,
        9100,  9160,  9209,  9477,  9495,  9517,  9656,  9708,  9991, 10036,
       10152],
      dtype='int64')

In [50]:
df_01_PB2_701 = df_01_PB2_701.reset_index(drop=True)

In [51]:
df_01_PB2_701.index

RangeIndex(start=0, stop=81, step=1)

In [52]:
df_01_PB2_701

,Unnamed: 0.2,Unnamed: 0.1,index,Unnamed: 0,Isolate_Id,PB2 Segment_Id,PB1 Segment_Id,PA Segment_Id,HA Segment_Id,NP Segment_Id,...,6_NA_ID,7_MP_ID,8_NS_ID,Host_ID,deduplication,Ambiguous_Count,Seq_Len,VADR_Screening_Passed,Codon_Pos_627,Codon_Pos_701
0,21,21,21,42,EPI_ISL_19660845,EPI3783462,EPI3783458,EPI3783451,EPI3783454,EPI3783455,...,EPI3783450,EPI3783463,EPI3783460,1,keep_earliest,0,2280,True,GAA,AAC
1,137,144,144,248,EPI_ISL_19792457,EPI4150060,EPI4150058,EPI4150055,EPI4150056,EPI4150057,...,EPI4150054,EPI4150061,EPI4150059,1,no_dup,0,2280,True,GAA,AAC
2,304,322,325,538,EPI_ISL_19825660,EPI4244023,EPI4244021,EPI4244018,EPI4244019,EPI4244020,...,EPI4244017,EPI4244024,EPI4244022,2,no_dup,1,2280,True,GAA,AAC
3,307,325,328,541,EPI_ISL_19825659,EPI4244015,EPI4244013,EPI4244010,EPI4244011,EPI4244012,...,EPI4244009,EPI4244016,EPI4244014,2,no_dup,2,2280,True,GAA,AAC
4,308,326,329,542,EPI_ISL_19825658,EPI4244007,EPI4244005,EPI4244002,EPI4244003,EPI4244004,...,EPI4244001,EPI4244008,EPI4244006,2,no_dup,0,2280,True,GAA,AAC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,9656,9903,9945,18043,EPI_ISL_19820567,EPI4226806,EPI4226802,EPI4226793,EPI4226797,EPI4226799,...,EPI4226791,EPI4226808,EPI4226804,2,no_dup,0,2280,True,GAA,AAC
77,9708,9957,9999,18157,EPI_ISL_17821770,EPI2603668,EPI2603669,EPI2603670,EPI2603671,EPI2603672,...,EPI2603673,EPI2603674,EPI2603675,2,no_dup,0,2280,True,GAA,AAC
78,9991,10247,10292,18753,EPI_ISL_19755944,EPI4035981,EPI4035979,EPI4035976,EPI4035977,EPI4035978,...,EPI4035975,EPI4035982,EPI4035980,1,no_dup,0,2280,True,GAA,AAC
79,10036,10294,10339,18869,EPI_ISL_19820795,EPI4228494,EPI4228490,EPI4228481,EPI4228485,EPI4228487,...,EPI4228480,EPI4228495,EPI4228492,1,keep_earliest,1,2280,True,GAA,AAC


In [53]:
df_01_PB2_701['Codon_Pos_701'].value_counts(dropna=False)

Codon_Pos_701
AAC    63
AAT    18
Name: count, dtype: int64

In [54]:
def get_cds_shap(sequence, shap_values, cds_coordinates, codon_start=1):
    """Count codons in the given coding sequence"""
    full_cds_sequence = ""

    # Step 1: Correctly parse ALL start and end coordinates from the string.
    coords = re.findall(r'[<>]?(\d+)\.\.[<>]?(\d+)', cds_coordinates)
    if not coords:
        raise ValueError(f"Could not parse CDS coordinates from string: {cds_coordinates}")

    # Step 2: Build the complete CDS by concatenating all exon parts.
    for start, end in coords:
        start_index, end_index = int(start) - 1, int(end)
        full_cds_sequence += sequence[start_index:end_index]
    shap_parts = [shap_values[:, int(start)-1:int(end)] for start, end in coords]

    full_shap_values = np.concatenate(shap_parts, axis=1)
    # Step 3: Apply the codon_start offset to the ASSEMBLED sequence.
    offset = int(codon_start) - 1
    trimmed_sequence = full_cds_sequence[offset:]
    trimmed_shape    = full_shap_values[:, offset:]

    # Step 4: Process the final sequence
    remainder = len(trimmed_sequence) % 3
    seq_length = len(trimmed_sequence) - remainder
    
    return trimmed_sequence[:seq_length], trimmed_shape[:, :seq_length]

In [55]:
x_onehot_pb2 = np.load('H5Nx_NorthAmerica_Shap/01_PB2/X_onehot_701_AAC_AAT.npy')

In [56]:
x_onehot_pb2.shape

(81, 5, 2400)

In [57]:
def get_codon_columns(df):
    """Return columns that look like codons (exactly 3 chars of A/T/G/C)."""
    return [c for c in df.columns if re.fullmatch(r'[ATGC]{3}', str(c))]
def is_at_rich(codon):
    # Define as A/T-rich if majority of bases are A or T
    return sum(1 for b in codon if b in 'AT') >= 2

In [58]:
def onehot_to_dna(onehot):
    
    alphabet = np.array(['A', 'C', 'G', 'T', 'N'])
    mask = onehot.any(axis=0)
    indices = np.argmax(onehot[:, mask], axis=0)
    dna_seq = "".join(alphabet[indices])
    
    return dna_seq

In [59]:
def find_differences_zero_based(seq1, seq2):
    """
    seq1, seq2: same-length DNA strings.
    Returns list of (index, base_in_seq1, base_in_seq2) with 0-based index.
    """
    diffs = []
    for i, (b1, b2) in enumerate(zip(seq1, seq2)):  # start=0 by default
        if b1 != b2:
            diffs.append((i, b1, b2))
    return diffs

In [60]:
# ==========================================
# 1. INITIALIZATION
# ==========================================
supplementary_data = []

# Constants for E627K (Codon 627)
TARGET_CODON_NUM = 701
# 1-based codon -> 0-based nucleotide index
# (627 - 1) * 3 = 1878
START_NUC_IDX = (TARGET_CODON_NUM - 1) * 3  

results = "/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results"
if not os.path.exists(results):
    os.makedirs(results)

print(f"Starting analysis...")
codon_list = np.load('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/codon_list.npy')
avi_medoid = np.load('/home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/01_PB2_avi_medoid_pre2020.npy')
dists = []
# ==========================================
# 2. MAIN PROCESSING LOOP
# ==========================================
for index, row in df_01_PB2_701.iterrows():
    seq_id = row['1_PB2_ID']
    
    # FILTER: Human + E627K
    n_ambiguous_bases = row['Ambiguous_Count']
    if n_ambiguous_bases <=10 and row['Codon_Pos_701']=='AAC' and row['Host_Group']=='Nonhuman Mammal': 
        
        seq_id = row['1_PB2_ID']
        isolate_name = row['Isolate_Name']
        sequence = fa_01_PB2[seq_id].seq.upper()
        clean_seq = re.sub(r'[^ACGT]', 'N', sequence.upper())
        
        assert clean_seq == onehot_to_dna(x_onehot_pb2[index])
        
        cds_coords = cds_coords_dict[seq_id]['polymerase PB2']
        extracted_coords = cds_coords['coordinate']
        codon_start = cds_coords.get('codon_start', 1)
        print (seq_id, cds_coords, codon_start)

        shap_scores_arr = []

        avg_codon_score = defaultdict(list)
        for fold in range(10):
            shap_scores_tmp = np.load(f'H5Nx_NorthAmerica_Shap/01_PB2/Shap_701_AAC_AAT_Fold_{fold}.npy')
            seq_scores_tmp = shap_scores_tmp[index, :, :, 2] * x_onehot_pb2[index, :, :]
            trimmed_seq, trimmed_shap = get_cds_shap(sequence, seq_scores_tmp, extracted_coords, codon_start)
            shap_scores_arr.append(trimmed_shap)
            codon_score = {}
            for pos, codon_idx in enumerate(range(0, len(trimmed_seq), 3)):
                codon = trimmed_seq[codon_idx:codon_idx+3]
                if 'N' not in codon and codon not in stop_codons:
                    codon_shap_score = np.sum(trimmed_shap[:, codon_idx:codon_idx+3])
                    codon_score[codon] = codon_score.get(codon,0.0) + codon_shap_score
            for key, value in codon_score.items():
                avg_codon_score[key].append(value)
        bar_data = {}      
        for codon in codon_list:
            if codon not in avg_codon_score:
                bar_data[codon] = 0.0
            else:
                bar_data[codon] = np.mean(np.asarray(avg_codon_score[codon]))
        # phenotypic_distance_check = cdist(np.array(list(bar_data.values())).reshape(1, -1), avi_medoid.reshape(1, -1), metric='cosine').flatten()
        # print (phenotypic_distance_check[0])
        phenotypic_distance = batch_cosine_dist(np.array(list(bar_data.values())).reshape(1, -1), avi_medoid)
        print (phenotypic_distance[0])
        dists.append(phenotypic_distance[0])

        
        stack_shap_score = np.stack(shap_scores_arr)
        mean_shap_score = np.mean(stack_shap_score, axis=0)
        print ('mean_shap_score shape:', mean_shap_score.shape)

        seq_scores = mean_shap_score[0:4, :]#raw_shap * raw_onehot
        
        # --- B. Get E627K Specific Scores (Positions 1, 2, 3) ---
        # START_NUC_IDX = 1878
        score_pos1 = np.sum(seq_scores[:, START_NUC_IDX])     # 1878 (A)
        score_pos2 = np.sum(seq_scores[:, START_NUC_IDX + 1]) # 1879 (A)
        score_pos3 = np.sum(seq_scores[:, START_NUC_IDX + 2]) # 1880 (G)
        
        total_701_score = score_pos1 + score_pos2 + score_pos3
        actual_codon_seq = trimmed_seq[START_NUC_IDX : START_NUC_IDX + 3] # Should be AAG
        print ('actual_codon_seq:', actual_codon_seq)

        # --- C. Get Top 10 Positive Nucleotides Globally ---
        # Sum axis 0 -> 1D array of scores per position
        per_nuc_scores = np.sum(seq_scores, axis=0) 
        
        positive_sites = []
        for i, score in enumerate(per_nuc_scores):
            if score > 0:
                # Store: (Position 1-based, Score, Nucleotide)
                positive_sites.append((i + 1, score, trimmed_seq[i]))
        
        # Sort descending
        positive_sites.sort(key=lambda x: x[1], reverse=True)
        top_20 = positive_sites[:20]
        
        # Format string: "Pos 1880(G): 0.52; Pos 120(A): 0.45"
        top_20_str = "; ".join([f"Pos {p}({n}): {s:.4f}" for p, s, n in top_20])


        # --- D. Build CSV Record ---
        record = {
            'Sequence_ID': seq_id,
            'Isolate_Name': row['Isolate_Name'],
            'Host': row['Host_Group'],
            'Mutation': 'D701N',
            'Codon_Seq': actual_codon_seq,
            'Phynotypic Distance:': round(phenotypic_distance[0], 4), 
            
            # 1. Specific Breakdown for 627
            'SHAP_Total_701': round(total_701_score, 4),
            'SHAP_701_Pos1_A': round(score_pos1, 4),
            'SHAP_701_Pos2_A': round(score_pos2, 4),
            'SHAP_701_Pos3_C': round(score_pos3, 4),
            
            # 2. Global Context
            'Top_20_Positive_Sites': top_20_str
        }
        supplementary_data.append(record)


        # --- E. Plotting ---
        # 1. Codon scores for scatter
        codon_shap_by_pos = defaultdict(float)
        
        for pos, codon_idx in enumerate(range(0, len(trimmed_seq), 3)):
            codon = trimmed_seq[codon_idx:codon_idx+3]
            if codon =='AAC' or codon=='AAT':
                codon_shap_score = np.sum(seq_scores[:, codon_idx:codon_idx+3])
                codon_shap_by_pos[pos+1] = codon_shap_score

        # 2. Logo Data: Transpose (4, L) -> (L, 4)
        seq_logo_data = np.transpose(seq_scores) 
        
        dummy_bar_data = bar_data

        plot_dashboard_grid(
            scatter_data=codon_shap_by_pos,
            logo_array=seq_logo_data,
            bar_data=dummy_bar_data,
            top_pos_data=top_20,
            codon=['Asparagine N', 'AAC', 'AAT'],
            fig_title=f"{seq_id} - {row['Isolate_Name']} - H5Nx North America",
            scatter_markers=[{'pos': 701, 'label': '701N', 'color': 'dodgerblue'}],
            logo_region=(2076, 2126),
            logo_highlight={'red': [(START_NUC_IDX, START_NUC_IDX + 3, "701N")]},
            subticks_frequency=5,
            is_at_rich_fn=is_at_rich,
            save_path=f"{results}/Mutation/H5Nx_NorthAmerica/701N_{seq_id}_H5Nx_NorthAmerica.png"
        )

# ==========================================
# 3. SAVE TO CSV
# ==========================================
df_supp = pd.DataFrame(supplementary_data)
# Sort by the 3rd position (G) score, or Total 627 score? Usually Total is best.
df_supp = df_supp.sort_values(by='SHAP_Total_701', ascending=False)

csv_name = f"{results}/Supplementary_Table_701N_H5Nx.csv"
df_supp.to_csv(csv_name, index=False)
print(f"Mean Phenotypic Distance: {np.mean(dists)}")
print(f"Std Dev: {np.std(dists)}")
print(f"Done. Saved table to {csv_name}")

Starting analysis...
EPI4244023 {'coordinate': '1..2280'} 1
1.7165573484037562
mean_shap_score shape: (5, 2280)
actual_codon_seq: AAC
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H5Nx_NorthAmerica/701N_EPI4244023_H5Nx_NorthAmerica.png
EPI4244015 {'coordinate': '1..2280'} 1
1.7200705394098852
mean_shap_score shape: (5, 2280)
actual_codon_seq: AAC
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H5Nx_NorthAmerica/701N_EPI4244015_H5Nx_NorthAmerica.png
EPI4244007 {'coordinate': '1..2280'} 1
1.7016826499819337
mean_shap_score shape: (5, 2280)
actual_codon_seq: AAC
Figure saved to: /home/hnguyen/Documents/PhD/Part2/Paper_02_Preparation/Github/IAV_Evolution/results/Mutation/H5Nx_NorthAmerica/701N_EPI4244007_H5Nx_NorthAmerica.png
EPI4253718 {'coordinate': '1..2280'} 1
1.720827853524757
mean_shap_score shape: (5, 2280)
actual_codon_seq: AAC
Figure saved to: /home/hnguyen